In [32]:
import json
from pathlib import Path
import sys

# Add project root to path to import forecaster
# Handle both cases: running from project root or from research directory
current_dir = Path().resolve()
if current_dir.name == 'research':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))

# Ensure we load the local forecaster package (not a cached/installed one)
import importlib
for mod in ['forecaster.models', 'forecaster']:
    if mod in sys.modules:
        del sys.modules[mod]
forecaster = importlib.import_module('forecaster')
from forecaster.models import MatchData

# Load all JSON files from the male directory
male_dir = project_root / 't20s_json' / 'male'
all_json_files = sorted(male_dir.glob('*.json'))
json_files = all_json_files

print(f"Found {len(all_json_files)} JSON files in male directory; reading {len(json_files)} files")

# Store all match data
all_matches = []

for json_file_path in json_files:
    try:
        with open(json_file_path, 'r') as f:
            json_data = json.load(f)
        
        # Convert to Python objects using Pydantic's model_validate
        match_data = MatchData.model_validate(json_data)
        all_matches.append(match_data)
    except Exception as e:
        print(f"Error processing {json_file_path.name}: {e}")

print(f"\nSuccessfully loaded {len(all_matches)} matches")

# Display info about first match as example
if all_matches:
    match_data = all_matches[0]
    print(f"\n--- Example: First Match ---")
    print(f"Match: {match_data.info.teams[0]} vs {match_data.info.teams[1]}")
    print(f"Date: {match_data.info.dates[0]}")
    print(f"Venue: {match_data.info.venue}")
    # print(f"Winner: {match_data.info.outcome.winner}")
    print(f"Number of innings: {len(match_data.innings)}")
    print(f"Total overs in first innings: {len(match_data.innings[0].overs)}")

Found 3112 JSON files in male directory; reading 3112 files

Successfully loaded 3112 matches

--- Example: First Match ---
Match: Australia vs Sri Lanka
Date: 2017-02-19
Venue: Simonds Stadium, South Geelong
Number of innings: 2
Total overs in first innings: 20


In [33]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

files_found = len(json_files) if 'json_files' in globals() else 0
matches_loaded = len(all_matches) if 'all_matches' in globals() else 0
errors = files_found - matches_loaded if files_found >= matches_loaded else 0

print(f"Files found: {files_found}")
print(f"Matches loaded: {matches_loaded}")
print(f"Errors: {errors}")

sample_count = min(3, matches_loaded)
if matches_loaded:
    print("\nSample matches:")
    for i in range(sample_count):
        md = all_matches[i]
        print(f"  {i+1}. {md.info.teams[0]} vs {md.info.teams[1]} at {md.info.venue} on {md.info.dates[0]}")

SUMMARY
Files found: 3112
Matches loaded: 3112
Errors: 0

Sample matches:
  1. Australia vs Sri Lanka at Simonds Stadium, South Geelong on 2017-02-19
  2. Australia vs Sri Lanka at Adelaide Oval on 2017-02-22
  3. Ireland vs Hong Kong at Bready Cricket Club, Magheramason on 2016-09-05


In [5]:
# Display JSON Schema Structure
import json
from pathlib import Path

# Load the schema template
schema_path = project_root / 'forecaster' / 'json_schema_template.json'

with open(schema_path, 'r') as f:
    schema = json.load(f)

# Pretty print the schema
print("=" * 80)
print("JSON STRUCTURE SCHEMA (without actual data)")
print("=" * 80)
print(json.dumps(schema, indent=2))


JSON STRUCTURE SCHEMA (without actual data)
{
  "meta": {
    "data_version": "string",
    "created": "string (YYYY-MM-DD)",
    "revision": "integer"
  },
  "info": {
    "balls_per_over": "integer (typically 6)",
    "city": "string",
    "dates": [
      "string (YYYY-MM-DD)"
    ],
    "gender": "string (e.g., 'male', 'female')",
    "match_type": "string (e.g., 'T20')",
    "match_type_number": "integer",
    "officials": {
      "match_referees": [
        "string"
      ],
      "tv_umpires": [
        "string"
      ],
      "umpires": [
        "string"
      ],
      "reserve_umpires": [
        "string (optional)"
      ]
    },
    "outcome": {
      "by": {
        "runs": "integer (optional)",
        "wickets": "integer (optional)"
      },
      "winner": "string (team name)"
    },
    "overs": "integer (typically 20 for T20)",
    "player_of_match": [
      "string (optional)"
    ],
    "players": {
      "TeamName1": [
        "string (player names)"
      ],
     

In [ ]:
# Break match_data into two innings and calculate scores

# Extract the two innings
first_innings = match_data.innings[0]
second_innings = match_data.innings[1]



MATCH SCORECARD

England: 179/8 (20 overs)
Australia: 79/10 (15 overs)

England won by 100 runs



In [7]:
# Additional statistics for each innings

def get_innings_statistics(innings):
    """Get detailed statistics for an innings"""
    stats = {
        'team': innings.team,
        'total_runs': 0,
        'total_wickets': 0,
        'total_balls': 0,
        'fours': 0,
        'sixes': 0,
        'extras': 0,
        'byes': 0,
        'legbyes': 0,
        'wides': 0,
        'noballs': 0
    }
    
    for over in innings.overs:
        for delivery in over.deliveries:
            stats['total_runs'] += delivery.runs.total
            stats['total_balls'] += 1
            
            # Count boundaries
            if delivery.runs.batter == 4:
                stats['fours'] += 1
            elif delivery.runs.batter == 6:
                stats['sixes'] += 1
            
            # Count wickets
            if delivery.wickets:
                stats['total_wickets'] += len(delivery.wickets)
            
            # Count extras
            if delivery.extras:
                stats['extras'] += delivery.runs.extras
                if 'byes' in delivery.extras:
                    stats['byes'] += delivery.extras['byes']
                if 'legbyes' in delivery.extras:
                    stats['legbyes'] += delivery.extras['legbyes']
                if 'wides' in delivery.extras:
                    stats['wides'] += delivery.extras['wides']
                if 'noballs' in delivery.extras:
                    stats['noballs'] += delivery.extras['noballs']
    
    # Calculate run rate
    stats['run_rate'] = (stats['total_runs'] / stats['total_balls'] * 6) if stats['total_balls'] > 0 else 0
    
    return stats

# Get statistics for both innings
first_innings_stats = get_innings_statistics(first_innings)
second_innings_stats = get_innings_statistics(second_innings)

# Display detailed statistics
print("\n" + "=" * 80)
print("DETAILED INNINGS STATISTICS")
print("=" * 80)

for stats in [first_innings_stats, second_innings_stats]:
    print(f"\n{stats['team']}:")
    print(f"  Total: {stats['total_runs']}/{stats['total_wickets']} in {stats['total_balls']} balls")
    print(f"  Run Rate: {stats['run_rate']:.2f}")
    print(f"  Boundaries: {stats['fours']} fours, {stats['sixes']} sixes")
    print(f"  Extras: {stats['extras']} ({stats['wides']} wides, {stats['noballs']} no-balls, "
          f"{stats['byes']} byes, {stats['legbyes']} leg-byes)")

print("\n" + "=" * 80)



DETAILED INNINGS STATISTICS

England:
  Total: 179/8 in 125 balls
  Run Rate: 8.59
  Boundaries: 19 fours, 3 sixes
  Extras: 6 (3 wides, 2 no-balls, 0 byes, 1 leg-byes)

Australia:
  Total: 79/10 in 90 balls
  Run Rate: 5.27
  Boundaries: 10 fours, 0 sixes
  Extras: 6 (1 wides, 2 no-balls, 1 byes, 2 leg-byes)



In [10]:
# Enhanced view: Show runs scored in each section (not just cumulative)

def get_innings_by_sections_with_runs_per_section(innings, total_overs, balls_per_over, section_size=2):
    """
    Break innings into sections and show both cumulative and runs per section.
    Adds balls remaining and boundary counts at the end of each section.
    """
    sections = []
    cumulative_runs = 0
    cumulative_wickets = 0
    cumulative_overs = 0
    current_section_overs = []
    section_runs = 0
    section_wickets = 0
    section_overs = 0
    cumulative_fours = 0
    cumulative_sixes = 0
    cumulative_dot_balls = 0
    section_fours = 0
    section_sixes = 0
    section_dot_balls = 0
    
    for over in innings.overs:
        # Process all deliveries in this over
        for delivery in over.deliveries:
            runs_in_delivery = delivery.runs.total
            cumulative_runs += runs_in_delivery
            section_runs += runs_in_delivery
            
            if runs_in_delivery == 0:
                cumulative_dot_balls += 1
                section_dot_balls += 1
            
            if delivery.runs.batter == 4:
                cumulative_fours += 1
                section_fours += 1
            elif delivery.runs.batter == 6:
                cumulative_sixes += 1
                section_sixes += 1
            
            if delivery.wickets:
                wickets_in_delivery = len(delivery.wickets)
                cumulative_wickets += wickets_in_delivery
                section_wickets += wickets_in_delivery
        
        current_section_overs.append(over.over)
        cumulative_overs += 1
        section_overs += 1
        
        # Check if we've completed a section (2 overs)
        if len(current_section_overs) == section_size:
            # Calculate run rates: runs / overs
            cumulative_run_rate = (cumulative_runs / cumulative_overs) if cumulative_overs > 0 else 0
            section_run_rate = (section_runs / section_overs) if section_overs > 0 else 0
            
            # Powerplay is first 6 overs (0-5), so powerplay_completed is True if overs_completed > 6
            powerplay_completed = (current_section_overs[-1] + 1) > 6
            
            overs_completed = current_section_overs[-1] + 1
            balls_remaining = max(total_overs - overs_completed, 0) * balls_per_over
            boundaries_last_2_overs = section_fours + section_sixes
            
            section_num = len(sections) + 1
            sections.append({
                'section': section_num,
                'overs': f"{current_section_overs[0]}-{current_section_overs[-1]}",
                'overs_completed': overs_completed,
                'runs_in_section': section_runs,
                'wickets_in_section': section_wickets,
                'section_run_rate': section_run_rate,
                'cumulative_runs': cumulative_runs,
                'cumulative_wickets': cumulative_wickets,
                'cumulative_run_rate': cumulative_run_rate,
                'powerplay_completed': powerplay_completed,
                'balls_remaining': balls_remaining,
                'fours_hit': cumulative_fours,
                'sixes_hit': cumulative_sixes,
                'boundaries_last_2_overs': boundaries_last_2_overs,
                'dot_balls_so_far': cumulative_dot_balls,
                'dot_balls_last_2_overs': section_dot_balls
            })
            current_section_overs = []
            section_runs = 0
            section_wickets = 0
            section_overs = 0
            section_fours = 0
            section_sixes = 0
            section_dot_balls = 0
    
    # Handle remaining overs
    if current_section_overs:
        cumulative_run_rate = (cumulative_runs / cumulative_overs) if cumulative_overs > 0 else 0
        section_run_rate = (section_runs / section_overs) if section_overs > 0 else 0
        powerplay_completed = (current_section_overs[-1] + 1) > 6
        overs_completed = current_section_overs[-1] + 1
        balls_remaining = max(total_overs - overs_completed, 0) * balls_per_over
        boundaries_last_2_overs = section_fours + section_sixes
        section_num = len(sections) + 1
        sections.append({
            'section': section_num,
            'overs': f"{current_section_overs[0]}-{current_section_overs[-1]}",
            'overs_completed': overs_completed,
            'runs_in_section': section_runs,
            'wickets_in_section': section_wickets,
            'section_run_rate': section_run_rate,
            'cumulative_runs': cumulative_runs,
            'cumulative_wickets': cumulative_wickets,
            'cumulative_run_rate': cumulative_run_rate,
            'powerplay_completed': powerplay_completed,
            'balls_remaining': balls_remaining,
            'fours_hit': cumulative_fours,
            'sixes_hit': cumulative_sixes,
            'boundaries_last_2_overs': boundaries_last_2_overs,
            'dot_balls_so_far': cumulative_dot_balls,
            'dot_balls_last_2_overs': section_dot_balls
        })
    
    return sections

# Get enhanced sections for both innings
first_innings_detailed = get_innings_by_sections_with_runs_per_section(
    first_innings, match_data.info.overs, match_data.info.balls_per_over
)
second_innings_detailed = get_innings_by_sections_with_runs_per_section(
    second_innings, match_data.info.overs, match_data.info.balls_per_over
)

# Display enhanced view
print("\n" + "=" * 130)
print("DETAILED SECTION BREAKDOWN (Runs per section + Cumulative + Run Rates)")
print("=" * 130)

for team_name, sections, innings_num in [
    (first_innings.team, first_innings_detailed, "1st"),
    (second_innings.team, second_innings_detailed, "2nd")
]:
    print(f"\n{team_name} ({innings_num} Innings):")
    print(f"{'Section':<10} {'Overs':<15} {'Runs':<8} {'Wkts':<6} {'RR':<8} {'Cum Runs':<12} {'Cum Wkts':<12} {'Cum RR':<10} {'PP Done':<10} {'Balls Rem':<10} {'Fours':<7} {'Sixes':<7} {'Bnds L2':<8} {'Dot so far':<12} {'Dot L2':<8}")
    print("-" * 170)
    for section in sections:
        print(f"{section['section']:<10} {section['overs']:<15} "
              f"{section['runs_in_section']:<8} {section['wickets_in_section']:<6} "
              f"{section['section_run_rate']:<8.2f} "
              f"{section['cumulative_runs']:<12} {section['cumulative_wickets']:<12} "
              f"{section['cumulative_run_rate']:<10.2f} {str(section['powerplay_completed']):<10} {section['balls_remaining']:<10} "
              f"{section['fours_hit']:<7} {section['sixes_hit']:<7} {section['boundaries_last_2_overs']:<8} "
              f"{section['dot_balls_so_far']:<12} {section['dot_balls_last_2_overs']:<8}")

print("\n" + "=" * 130)



DETAILED SECTION BREAKDOWN (Runs per section + Cumulative + Run Rates)

England (1st Innings):
Section    Overs           Runs     Wkts   RR       Cum Runs     Cum Wkts     Cum RR     PP Done    Balls Rem  Fours   Sixes   Bnds L2  Dot so far   Dot L2  
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
1          0-1             6        0      3.00     6            0            3.00       False      108        0       0       0        9            9       
2          2-3             23       1      11.50    29           1            7.25       False      96         5       0       5        13           4       
3          4-5             21       1      10.50    50           2            8.33       False      84         7       0       2        15           2       
4          6-7             18       0      9.00     68           2            8.50       True       7